In [1]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/NLP_assignment/'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/NLP_assignment/']


In [3]:
from millionaire_client import MillionaireClient, AuthenticationError

In [ ]:
API_URL = "http://131.175.15.22:51111/"
username = ""
password = ""

In [5]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, gary! (Role: student)


In [6]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


In [7]:
# Choose a competition ID
comp_id = 1

In [20]:
def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [21]:
import json
import re
import html
from urllib.parse import quote, urlencode, urljoin, urlparse, parse_qs
from urllib.error import HTTPError
from urllib.request import Request, urlopen
import time

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_USER_AGENT = "PoliMillionaireNLP/1.0 student project"
WIKIPEDIA_REQUEST_DELAY_SECONDS = 0.8
WIKIPEDIA_429_BACKOFF_SECONDS = 4.0
WIKIPEDIA_MAX_RETRIES = 2
MAX_WIKIPEDIA_SEARCH_QUERIES = 2
_LAST_WIKIPEDIA_REQUEST_TIME = 0.0
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that",
    "the", "their", "there", "these", "this", "those", "to", "was", "were",
    "what", "when", "where", "which", "who", "why", "with", "according", "article",
    "considered", "important", "goal", "goals", "main", "primary", "following"
}


def question_to_text(question) -> str:
    """Accept a string, a dict, or a millionaire_client Question object."""
    if hasattr(question, "text"):
        return str(question.text)
    if isinstance(question, dict) and "text" in question:
        return str(question["text"])
    return str(question)


def normalize_wikipedia_text(text: str) -> str:
    """Clean a plain Wikipedia extract enough for later NLP steps."""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text: str) -> list[str]:
    return [token for token in re.findall(r"[a-z0-9]+", str(text).lower()) if len(token) > 1]


def expand_term(token: str) -> set[str]:
    """Tiny synonym/variant helper for common historical wording traps."""
    variants = {token}
    if token == "roman":
        variants.update({"rome", "romans"})
    elif token in {"rome", "romans"}:
        variants.add("roman")
    return variants


def extract_keywords(text: str, limit: int = 10) -> list[str]:
    keywords = []
    seen = set()
    for token in tokenize(text):
        if token in STOPWORDS or token in seen:
            continue
        keywords.append(token)
        seen.add(token)
        if len(keywords) >= limit:
            break
    return keywords


def capital_context_phrases(question_text: str) -> list[str]:
    """Build focused phrases like 'Roman marriage' from capitalized topic words."""
    words = re.findall(r"[A-Za-z][A-Za-z'-]*", question_text)
    phrases = []
    for index, word in enumerate(words):
        if not word[:1].isupper() or word.lower() in STOPWORDS:
            continue
        phrase_words = [word]
        for next_word in words[index + 1:index + 4]:
            if next_word.lower() in STOPWORDS:
                break
            phrase_words.append(next_word)
        if len(phrase_words) > 1:
            phrases.append(" ".join(phrase_words))
    return phrases


def build_wikipedia_search_queries(question) -> list[str]:
    """Create several focused Wikipedia queries instead of trusting the full question only."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = normalize_wikipedia_text(cleaned)
    keywords = extract_keywords(cleaned, limit=10)

    queries = []
    queries.extend(capital_context_phrases(cleaned))
    if keywords:
        queries.append(" ".join(keywords[:6]))
    if len(keywords) >= 2:
        queries.append(" ".join(keywords[:2]))
    queries.append(cleaned)
    queries.append(question_text)

    deduped = []
    seen = set()
    for query in queries:
        normalized = normalize_wikipedia_text(query).lower()
        if normalized and normalized not in seen:
            deduped.append(query)
            seen.add(normalized)
    return deduped


def wikipedia_request(params: dict, timeout: float = 6.0) -> dict:
    """Call the free MediaWiki API with delay and simple 429 backoff."""
    global _LAST_WIKIPEDIA_REQUEST_TIME

    url = f"{WIKIPEDIA_API}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": WIKIPEDIA_USER_AGENT})

    for attempt in range(WIKIPEDIA_MAX_RETRIES + 1):
        elapsed_since_last = time.monotonic() - _LAST_WIKIPEDIA_REQUEST_TIME
        sleep_for = WIKIPEDIA_REQUEST_DELAY_SECONDS - elapsed_since_last
        if sleep_for > 0:
            time.sleep(sleep_for)

        try:
            with urlopen(request, timeout=timeout) as response:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()
                return json.loads(response.read().decode("utf-8"))
        except HTTPError as exc:
            _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()
            if exc.code != 429 or attempt >= WIKIPEDIA_MAX_RETRIES:
                raise

            retry_after = exc.headers.get("Retry-After")
            try:
                wait_seconds = float(retry_after) if retry_after else WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)
            except ValueError:
                wait_seconds = WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)

            print(f"Wikipedia rate limit hit. Waiting {wait_seconds:.1f}s before retry {attempt + 1}/{WIKIPEDIA_MAX_RETRIES}...")
            time.sleep(wait_seconds)


def search_wikipedia(query: str, limit: int = 5, timeout: float = 6.0) -> list[dict]:
    """Search Wikipedia and return candidate pages for one query string."""
    query = normalize_wikipedia_text(query)
    data = wikipedia_request(
        {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": limit,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
    )

    results = data.get("query", {}).get("search", [])
    return [
        {
            "title": item.get("title", ""),
            "page_id": item.get("pageid"),
            "snippet": normalize_wikipedia_text(re.sub(r"<[^>]+>", " ", item.get("snippet", ""))),
            "query": query,
            "search_rank": rank,
        }
        for rank, item in enumerate(results, start=1)
    ]


def collect_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None) -> list[dict]:
    """Search capped focused queries and deduplicate candidate pages by title."""
    candidates_by_title = {}
    search_queries = build_wikipedia_search_queries(question)
    if max_search_queries is None:
        max_search_queries = MAX_WIKIPEDIA_SEARCH_QUERIES
    if max_search_queries is not None:
        search_queries = search_queries[:max_search_queries]
    for query in search_queries:
        try:
            results = search_wikipedia(query, limit=per_query_limit, timeout=timeout)
        except Exception as exc:
            print(f"Wikipedia search skipped for {query!r}: {exc}")
            continue

        for result in results:
            title_key = result["title"].lower()
            if title_key not in candidates_by_title:
                candidates_by_title[title_key] = result
            else:
                candidates_by_title[title_key]["search_rank"] = min(
                    candidates_by_title[title_key]["search_rank"],
                    result["search_rank"],
                )
    return list(candidates_by_title.values())


def candidate_relevance_score(candidate: dict, question) -> float:
    """Score title/snippet overlap with question keywords; penalize very generic one-word titles."""
    question_text = question_to_text(question)
    keywords = extract_keywords(question_text, limit=10)
    candidate_text = f"{candidate.get('title', '')} {candidate.get('snippet', '')}"
    candidate_terms = set(tokenize(candidate_text))

    matched = 0
    for keyword in keywords:
        if expand_term(keyword) & candidate_terms:
            matched += 1

    overlap = matched / max(1, len(keywords))
    title_terms = tokenize(candidate.get("title", ""))
    rank_bonus = 1.0 / max(1, candidate.get("search_rank", 1))
    generic_penalty = 0.35 if len(title_terms) == 1 and len(keywords) > 1 else 0.0

    return (1.6 * overlap) + (0.25 * rank_bonus) - generic_penalty


def rank_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None) -> list[dict]:
    candidates = collect_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries)
    for candidate in candidates:
        candidate["candidate_score"] = candidate_relevance_score(candidate, question)
    return sorted(candidates, key=lambda item: item["candidate_score"], reverse=True)


def fetch_wikipedia_extract(title: str, timeout: float = 6.0) -> dict:
    """Fetch a Wikipedia page as a plain-text document."""
    data = wikipedia_request(
        {
            "action": "query",
            "prop": "extracts|info",
            "explaintext": 1,
            "exsectionformat": "plain",
            "inprop": "url",
            "titles": title,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
    )

    pages = data.get("query", {}).get("pages", {})
    page = next(iter(pages.values()), {}) if pages else {}
    return {
        "title": page.get("title", title),
        "page_id": page.get("pageid"),
        "url": page.get("fullurl") or f"https://en.wikipedia.org/wiki/{quote(title.replace(' ', '_'))}",
        "text": normalize_wikipedia_text(page.get("extract", "")),
        "source": "Wikipedia",
    }


def get_wikipedia_documents_for_question(question, top_n: int = 5, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None) -> list[dict]:
    """
    Return the top N related Wikipedia documents for a question.

    This avoids the trap of trusting only Wikipedia's first result for the full question.
    """
    query = question_to_text(question)
    candidates = rank_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries)
    documents = []

    for candidate in candidates[:top_n]:
        try:
            document = fetch_wikipedia_extract(candidate["title"], timeout=timeout)
        except Exception as exc:
            print(f"Wikipedia page skipped for {candidate['title']!r}: {exc}")
            continue

        document["query"] = query
        document["matched_query"] = candidate.get("query")
        document["search_rank"] = candidate.get("search_rank")
        document["candidate_score"] = candidate.get("candidate_score", 0.0)
        document["snippet"] = candidate.get("snippet", "")
        document["search_results"] = candidates
        documents.append(document)

    return documents


def get_wikipedia_document_for_question(question, search_limit: int = 5, timeout: float = 6.0) -> dict:
    """Backward-compatible helper: return only the highest-ranked document."""
    documents = get_wikipedia_documents_for_question(question, top_n=1, per_query_limit=search_limit, timeout=timeout)
    if documents:
        return documents[0]
    return {
        "query": question_to_text(question),
        "title": None,
        "page_id": None,
        "url": None,
        "text": "",
        "source": "Wikipedia",
        "search_results": [],
    }


# ---- Optional extra history/culture sources for the RAG document pool ----
# These helpers return the same document shape as the Wikipedia helper:
# {source, title, url, text, query, matched_query, candidate_score, snippet}

EXTRA_SOURCE_USER_AGENT = WIKIPEDIA_USER_AGENT
EXTRA_SOURCE_REQUEST_DELAY_SECONDS = 0.15
_LAST_EXTRA_SOURCE_REQUEST_TIME = {}

MEDIAWIKI_REST_SEARCH_API = "https://en.wikipedia.org/w/rest.php/v1/search/page"
MEDIAWIKI_REST_SUMMARY_API = "https://en.wikipedia.org/api/rest_v1/page/summary"
LOC_SEARCH_API = "https://www.loc.gov/search/"
MET_SEARCH_API = "https://collectionapi.metmuseum.org/public/collection/v1/search"
MET_OBJECT_API = "https://collectionapi.metmuseum.org/public/collection/v1/objects"
EUROPEANA_SEARCH_API = "https://api.europeana.eu/record/v2/search.json"
NARA_SEARCH_API = "https://catalog.archives.gov/api/v2/records/search"
FORDHAM_SEARCH_API = "https://search.fordham.edu/s/search.html"
SCAIFE_SEARCH_API = "https://scaife.perseus.org/search/json/"


def clean_source_text(text: str) -> str:
    """Clean snippets from JSON or HTML sources into compact plain text."""
    text = html.unescape(str(text or "")).replace("\xa0", " ")
    text = re.sub(r"(?is)<(script|style|noscript).*?</\1>", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def html_page_to_text(raw_html: str) -> str:
    raw_html = re.sub(r"(?is)<(header|footer|nav|script|style|noscript).*?</\1>", " ", str(raw_html or ""))
    return clean_source_text(raw_html)


def iter_text_values(value, max_items: int = 20):
    """Flatten common API JSON values into text strings."""
    if value is None or max_items <= 0:
        return []
    if isinstance(value, str):
        text = clean_source_text(value)
        return [text] if text else []
    if isinstance(value, (int, float)):
        return [str(value)]
    if isinstance(value, dict):
        values = []
        for item in value.values():
            values.extend(iter_text_values(item, max_items=max_items - len(values)))
            if len(values) >= max_items:
                break
        return values
    if isinstance(value, (list, tuple, set)):
        values = []
        for item in value:
            values.extend(iter_text_values(item, max_items=max_items - len(values)))
            if len(values) >= max_items:
                break
        return values
    text = clean_source_text(value)
    return [text] if text else []


def join_text_fields(*values, max_chars: int = 3500) -> str:
    parts = []
    seen = set()
    for value in values:
        for text in iter_text_values(value):
            key = text.lower()
            if text and key not in seen:
                parts.append(text)
                seen.add(key)
            if sum(len(part) for part in parts) >= max_chars:
                break
    return clean_source_text(". ".join(parts))[:max_chars]


def source_json_request(url: str, timeout: float = 4.0, source_name: str = "source", headers=None, delay=None) -> dict:
    global _LAST_EXTRA_SOURCE_REQUEST_TIME

    if delay is None:
        delay = EXTRA_SOURCE_REQUEST_DELAY_SECONDS
    last_time = _LAST_EXTRA_SOURCE_REQUEST_TIME.get(source_name, 0.0)
    sleep_for = delay - (time.monotonic() - last_time)
    if sleep_for > 0:
        time.sleep(sleep_for)

    request_headers = {
        "User-Agent": EXTRA_SOURCE_USER_AGENT,
        "Accept": "application/json,text/plain,*/*",
    }
    if headers:
        request_headers.update(headers)

    request = Request(url, headers=request_headers)
    with urlopen(request, timeout=timeout) as response:
        _LAST_EXTRA_SOURCE_REQUEST_TIME[source_name] = time.monotonic()
        raw_text = response.read().decode("utf-8", "replace")
        content_type = response.headers.get("Content-Type", "").lower()
        if "json" not in content_type and not raw_text.lstrip().startswith(("{", "[")):
            raise ValueError(f"{source_name} did not return JSON; content-type={content_type!r}")
        return json.loads(raw_text)


def source_text_request(url: str, timeout: float = 4.0, source_name: str = "source", headers=None, delay=None) -> str:
    global _LAST_EXTRA_SOURCE_REQUEST_TIME

    if delay is None:
        delay = EXTRA_SOURCE_REQUEST_DELAY_SECONDS
    last_time = _LAST_EXTRA_SOURCE_REQUEST_TIME.get(source_name, 0.0)
    sleep_for = delay - (time.monotonic() - last_time)
    if sleep_for > 0:
        time.sleep(sleep_for)

    request_headers = {"User-Agent": EXTRA_SOURCE_USER_AGENT, "Accept": "text/html,text/plain,*/*"}
    if headers:
        request_headers.update(headers)

    request = Request(url, headers=request_headers)
    with urlopen(request, timeout=timeout) as response:
        _LAST_EXTRA_SOURCE_REQUEST_TIME[source_name] = time.monotonic()
        return response.read().decode("utf-8", "replace")


def build_history_source_query(question, max_terms: int = 7) -> str:
    """Build one compact query for non-Wikipedia sources."""
    question_text = question_to_text(question)
    phrases = capital_context_phrases(question_text)
    keywords = extract_keywords(question_text, limit=max_terms)
    if phrases:
        phrase_terms = set(tokenize(phrases[0]))
        tail = [keyword for keyword in keywords if keyword not in phrase_terms]
        query = " ".join([phrases[0], *tail[:max_terms]])
    elif keywords:
        query = " ".join(keywords[:max_terms])
    else:
        query = question_text
    return clean_source_text(query)


def source_relevance_score(question, title: str, text: str, rank: int = 1) -> float:
    candidate = {"title": title or "", "snippet": (text or "")[:800], "search_rank": rank}
    return candidate_relevance_score(candidate, question)


def make_source_document(source: str, title: str, url: str, text: str, query: str, question, rank: int = 1, snippet: str = "") -> dict:
    text = clean_source_text(text)
    snippet = clean_source_text(snippet or text[:500])
    return {
        "source": source,
        "title": clean_source_text(title) or source,
        "url": url,
        "text": text,
        "query": question_to_text(question),
        "matched_query": query,
        "search_rank": rank,
        "candidate_score": source_relevance_score(question, title, f"{snippet} {text}", rank=rank),
        "snippet": snippet,
    }


def first_text(value, default: str = "") -> str:
    values = iter_text_values(value, max_items=1)
    return values[0] if values else default


def get_mediawiki_rest_documents_for_question(question, top_n: int = 1, timeout: float = 4.0, fetch_summary: bool = True) -> list[dict]:
    query = build_history_source_query(question)
    data = source_json_request(
        f"{MEDIAWIKI_REST_SEARCH_API}?{urlencode({'q': query, 'limit': top_n})}",
        timeout=timeout,
        source_name="MediaWiki REST",
    )
    documents = []
    for rank, page in enumerate(data.get("pages", [])[:top_n], start=1):
        title = clean_source_text(page.get("title") or page.get("key") or "Wikipedia page")
        key = page.get("key") or title.replace(" ", "_")
        page_url = f"https://en.wikipedia.org/wiki/{quote(str(key), safe=':/_')}"
        summary_text = ""
        if fetch_summary:
            try:
                summary = source_json_request(
                    f"{MEDIAWIKI_REST_SUMMARY_API}/{quote(str(key), safe='')}",
                    timeout=timeout,
                    source_name="MediaWiki REST",
                )
                summary_text = summary.get("extract", "")
                page_url = summary.get("content_urls", {}).get("desktop", {}).get("page", page_url)
            except Exception as exc:
                print(f"MediaWiki REST summary skipped for {title!r}: {exc}")
        text = join_text_fields(title, page.get("description"), page.get("excerpt"), summary_text)
        if text:
            documents.append(make_source_document("MediaWiki REST", title, page_url, text, query, question, rank=rank, snippet=page.get("excerpt", "")))
    return documents


def get_loc_documents_for_question(question, top_n: int = 1, timeout: float = 4.0) -> list[dict]:
    query = build_history_source_query(question)
    data = source_json_request(
        f"{LOC_SEARCH_API}?{urlencode({'fo': 'json', 'q': query, 'c': top_n})}",
        timeout=timeout,
        source_name="Library of Congress",
    )
    documents = []
    for rank, item in enumerate(data.get("results", [])[:top_n], start=1):
        title = first_text(item.get("title"), "Library of Congress result")
        url = item.get("url") or item.get("id") or "https://www.loc.gov/"
        text = join_text_fields(
            item.get("title"),
            item.get("date"),
            item.get("description"),
            item.get("subject"),
            item.get("location"),
            item.get("notes"),
            item.get("item", {}).get("description") if isinstance(item.get("item"), dict) else None,
        )
        if text:
            documents.append(make_source_document("Library of Congress", title, url, text, query, question, rank=rank))
    return documents


def get_met_collection_documents_for_question(question, top_n: int = 1, timeout: float = 4.0) -> list[dict]:
    query = build_history_source_query(question, max_terms=5)
    search = source_json_request(
        f"{MET_SEARCH_API}?{urlencode({'q': query})}",
        timeout=timeout,
        source_name="Met Collection",
    )
    object_ids = search.get("objectIDs") or []
    documents = []
    for rank, object_id in enumerate(object_ids[:top_n], start=1):
        try:
            item = source_json_request(
                f"{MET_OBJECT_API}/{object_id}",
                timeout=timeout,
                source_name="Met Collection",
            )
        except Exception as exc:
            print(f"Met object skipped for {object_id!r}: {exc}")
            continue
        tags = [tag.get("term") for tag in item.get("tags") or [] if isinstance(tag, dict)]
        title = first_text(item.get("title"), f"Met object {object_id}")
        text = join_text_fields(
            item.get("title"), item.get("objectName"), item.get("culture"), item.get("period"),
            item.get("dynasty"), item.get("reign"), item.get("objectDate"), item.get("artistDisplayName"),
            item.get("artistDisplayBio"), item.get("medium"), item.get("department"), item.get("classification"),
            item.get("geographyType"), item.get("city"), item.get("state"), item.get("country"),
            item.get("region"), item.get("subregion"), item.get("excavation"), tags,
        )
        if text:
            documents.append(make_source_document("Met Collection", title, item.get("objectURL") or f"{MET_OBJECT_API}/{object_id}", text, query, question, rank=rank))
    return documents


def get_europeana_documents_for_question(question, api_key: str = "", top_n: int = 1, timeout: float = 4.0) -> list[dict]:
    if not api_key:
        return []
    query = build_history_source_query(question)
    data = source_json_request(
        f"{EUROPEANA_SEARCH_API}?{urlencode({'wskey': api_key, 'query': query, 'rows': top_n})}",
        timeout=timeout,
        source_name="Europeana",
    )
    documents = []
    for rank, item in enumerate(data.get("items", [])[:top_n], start=1):
        title = first_text(item.get("title"), "Europeana result")
        url = item.get("guid") or item.get("link") or "https://www.europeana.eu/"
        text = join_text_fields(
            item.get("title"), item.get("dcDescription"), item.get("dataProvider"), item.get("provider"),
            item.get("type"), item.get("year"), item.get("dcSubject"), item.get("country"), item.get("edmPlace"),
        )
        if text:
            documents.append(make_source_document("Europeana", title, url, text, query, question, rank=rank))
    return documents


def nara_candidate_records(data: dict) -> list:
    containers = [data]
    if isinstance(data, dict) and isinstance(data.get("body"), dict):
        containers.append(data["body"])
    for container in containers:
        for key in ("results", "records"):
            value = container.get(key) if isinstance(container, dict) else None
            if isinstance(value, list):
                return value
        hits = container.get("hits") if isinstance(container, dict) else None
        if isinstance(hits, list):
            return hits
        if isinstance(hits, dict):
            for key in ("hits", "records", "results"):
                value = hits.get(key)
                if isinstance(value, list):
                    return value
    return []


def get_national_archives_documents_for_question(question, api_key: str = "", top_n: int = 1, timeout: float = 4.0) -> list[dict]:
    if not api_key:
        return []
    query = build_history_source_query(question)
    headers = {"x-api-key": api_key, "Content-Type": "application/json"}
    data = source_json_request(
        f"{NARA_SEARCH_API}?{urlencode({'q': query, 'rows': top_n})}",
        timeout=timeout,
        source_name="National Archives",
        headers=headers,
    )
    documents = []
    for rank, hit in enumerate(nara_candidate_records(data)[:top_n], start=1):
        record = hit.get("_source") if isinstance(hit, dict) else hit
        if isinstance(record, dict) and isinstance(record.get("description"), dict):
            description = record["description"]
        elif isinstance(record, dict):
            description = record
        else:
            description = {"text": record}
        title = first_text(description.get("title") or description.get("itemTitle") or record.get("title") if isinstance(record, dict) else None, "National Archives result")
        naid = first_text(description.get("naId") or description.get("naid") or description.get("identifierNaid"))
        url = f"https://catalog.archives.gov/id/{naid}" if naid else "https://catalog.archives.gov/"
        text = join_text_fields(
            description.get("title"), description.get("scopeAndContentNote"), description.get("scopeContent"),
            description.get("productionDateArray"), description.get("inclusiveStartDate"), description.get("inclusiveEndDate"),
            description.get("levelOfDescription"), description.get("typeOfMaterials"), description.get("generalRecordsTypeArray"),
        )
        if text:
            documents.append(make_source_document("National Archives", title, url, text, query, question, rank=rank))
    return documents


def fordham_result_urls(search_html: str, limit: int = 3) -> list[tuple[str, str]]:
    results = []
    seen = set()
    for match in re.finditer(r'<a[^>]+href=["\']([^"\']+)["\'][^>]*>(.*?)</a>', search_html, flags=re.IGNORECASE | re.DOTALL):
        href = html.unescape(match.group(1))
        label = clean_source_text(match.group(2))
        if "/s/redirect" not in href or "url=" not in href:
            continue
        full_href = urljoin("https://search.fordham.edu", href)
        actual_url = parse_qs(urlparse(full_href).query).get("url", [""])[0]
        if "sourcebooks" not in actual_url.lower() or actual_url in seen:
            continue
        seen.add(actual_url)
        results.append((actual_url, label or "Fordham Sourcebooks result"))
        if len(results) >= limit:
            break
    return results


def get_fordham_sourcebooks_documents_for_question(question, top_n: int = 1, timeout: float = 4.0) -> list[dict]:
    query = build_history_source_query(question)
    search_url = f"{FORDHAM_SEARCH_API}?{urlencode({'query': query, 'collection': 'fordham~sp-search', 'clive': 'fordham~ds-sourcebooks', 'num_ranks': max(10, top_n)})}"
    search_html = source_text_request(search_url, timeout=timeout, source_name="Fordham Sourcebooks")
    documents = []
    for rank, (url, label) in enumerate(fordham_result_urls(search_html, limit=top_n), start=1):
        try:
            page_html = source_text_request(url, timeout=timeout, source_name="Fordham Sourcebooks")
        except Exception as exc:
            print(f"Fordham page skipped for {url!r}: {exc}")
            continue
        text = html_page_to_text(page_html)
        if len(text) > 4500:
            text = text[:4500]
        if text:
            documents.append(make_source_document("Fordham Sourcebooks", label, url, text, query, question, rank=rank, snippet=text[:500]))
    return documents


def get_perseus_scaife_documents_for_question(question, top_n: int = 1, timeout: float = 4.0, search_kind: str = "form", results_format: str = "instances") -> list[dict]:
    query = build_history_source_query(question, max_terms=4)
    data = source_json_request(
        f"{SCAIFE_SEARCH_API}?{urlencode({'kind': search_kind, 'format': results_format, 'q': query, 'page_num': 1, 'type': 'library'})}",
        timeout=timeout,
        source_name="Perseus Scaife",
    )
    documents = []
    for rank, item in enumerate(data.get("results", [])[:top_n], start=1):
        passage = item.get("passage", {}) if isinstance(item, dict) else {}
        text_meta = passage.get("text", {}) if isinstance(passage.get("text"), dict) else {}
        ancestors = text_meta.get("ancestors") or []
        author = first_text(ancestors[0].get("label") if ancestors and isinstance(ancestors[0], dict) else "")
        work = first_text(text_meta.get("label"), "Perseus passage")
        urn = passage.get("urn", "")
        title = clean_source_text(" - ".join(part for part in [author, work, urn] if part))
        content = join_text_fields(item.get("content"), text_meta.get("human_lang"), text_meta.get("kind"), urn)
        url = urljoin("https://scaife.perseus.org", passage.get("url") or text_meta.get("url") or "/")
        if content:
            documents.append(make_source_document("Perseus Scaife", title, url, content, query, question, rank=rank))
    return documents


def get_secret_value(name: str, default: str = "") -> str:
    """Read a secret from environment variables or Colab Secrets."""
    import os

    value = os.environ.get(name)
    if value:
        return value

    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass

    return default


_REDDIT_CLIENT_CACHE = {}


def get_reddit_client(client_id: str = "", client_secret: str = "", user_agent: str = ""):
    """Create a read-only PRAW client for Reddit search."""
    client_id = client_id or get_secret_value("REDDIT_CLIENT_ID")
    client_secret = client_secret or get_secret_value("REDDIT_CLIENT_SECRET")
    user_agent = user_agent or get_secret_value("REDDIT_USER_AGENT")

    if not client_id or not client_secret:
        print("Reddit skipped: set REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET in Colab Secrets/env vars or the game hyperparameters.")
        return None
    if not user_agent:
        user_agent = "python:nlp-entertainment-rag:v1.0 (by /u/YOUR_REDDIT_USERNAME)"
        print("Reddit warning: using placeholder user_agent. Replace REDDIT_USER_AGENT with your Reddit username.")

    cache_key = (client_id, client_secret, user_agent)
    if cache_key in _REDDIT_CLIENT_CACHE:
        return _REDDIT_CLIENT_CACHE[cache_key]

    try:
        import praw
    except ImportError:
        print("Reddit skipped: install praw first with `pip install praw`, or run the actual game cell so it installs dependencies.")
        return None

    reddit = praw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent,
        check_for_async=False,
    )
    _REDDIT_CLIENT_CACHE[cache_key] = reddit
    return reddit


def clean_reddit_body(text: str) -> str:
    text = clean_source_text(text)
    if text.lower() in {"[deleted]", "[removed]"}:
        return ""
    return text


def build_reddit_search_query(question, max_terms: int = 8) -> str:
    """Make a compact entertainment-friendly Reddit query from the question only."""
    query = build_history_source_query(question, max_terms=max_terms)
    query = re.sub(r"\b(which|what|who|when|where|following|best|describes)\b", " ", query, flags=re.IGNORECASE)
    query = clean_source_text(query)
    return query or question_to_text(question)


def reddit_submission_to_document(post, subreddit_name: str, query: str, question, rank: int, include_comments: bool = False, comment_limit: int = 0) -> dict:
    title = clean_reddit_body(getattr(post, "title", ""))
    selftext = clean_reddit_body(getattr(post, "selftext", ""))
    parts = [title, selftext]

    if include_comments and comment_limit > 0:
        try:
            post.comment_sort = "confidence"
            post.comments.replace_more(limit=0)
            comments = []
            for comment in post.comments[:comment_limit]:
                body = clean_reddit_body(getattr(comment, "body", ""))
                if body:
                    comments.append(body)
            if comments:
                parts.append("Top Reddit comments: " + " ".join(comments))
        except Exception as exc:
            print(f"Reddit comments skipped for {title!r}: {exc}")

    text = join_text_fields(*parts, max_chars=3500)
    permalink = getattr(post, "permalink", "")
    url = "https://www.reddit.com" + permalink if permalink else getattr(post, "url", "")
    source = f"Reddit r/{subreddit_name}"
    score = source_relevance_score(question, title, text, rank=rank)
    reddit_score = getattr(post, "score", 0) or 0
    comment_count = getattr(post, "num_comments", 0) or 0
    score += min(0.25, max(0, reddit_score) / 20000) + min(0.15, max(0, comment_count) / 5000)

    doc = make_source_document(source, title, url, text, query, question, rank=rank, snippet=text[:500])
    doc["candidate_score"] = score
    doc["reddit_score"] = reddit_score
    doc["reddit_num_comments"] = comment_count
    return doc


def get_reddit_documents_for_question(
    question,
    client_id: str = "",
    client_secret: str = "",
    user_agent: str = "",
    subreddits=None,
    top_n_per_subreddit: int = 1,
    max_docs: int = 4,
    sort: str = "relevance",
    time_filter: str = "all",
    include_comments: bool = False,
    comment_limit: int = 0,
) -> list[dict]:
    """Search Reddit posts and return them as RAG documents."""
    reddit = get_reddit_client(client_id=client_id, client_secret=client_secret, user_agent=user_agent)
    if reddit is None:
        return []

    if subreddits is None:
        subreddits = ["movies", "television", "gaming", "Music", "popculturechat"]
    if isinstance(subreddits, str):
        subreddits = [item.strip() for item in subreddits.split(",") if item.strip()]

    query = build_reddit_search_query(question)
    documents = []
    for subreddit_name in subreddits:
        if max_docs and len(documents) >= max_docs:
            break
        try:
            subreddit = reddit.subreddit(subreddit_name)
            posts = subreddit.search(query, sort=sort, time_filter=time_filter, limit=top_n_per_subreddit)
            for rank, post in enumerate(posts, start=1):
                if max_docs and len(documents) >= max_docs:
                    break
                doc = reddit_submission_to_document(
                    post,
                    subreddit_name=subreddit_name,
                    query=query,
                    question=question,
                    rank=rank,
                    include_comments=include_comments,
                    comment_limit=comment_limit,
                )
                if doc.get("text"):
                    documents.append(doc)
        except Exception as exc:
            print(f"Reddit source skipped for r/{subreddit_name}: {exc}")

    documents.sort(key=lambda item: item.get("candidate_score", 0.0), reverse=True)
    return documents[:max_docs] if max_docs else documents


def dedupe_documents(documents: list[dict]) -> list[dict]:
    unique = []
    seen = set()
    for doc in documents:
        key = (doc.get("source", ""), doc.get("url") or doc.get("title") or "")
        if key in seen:
            continue
        seen.add(key)
        unique.append(doc)
    return unique


def get_multi_source_documents_for_question(
    question,
    top_n: int = 2,
    per_query_limit: int = 2,
    timeout: float = 3.0,
    max_search_queries=None,
    use_wikipedia: bool = True,
    extra_source_top_n: int = 1,
    max_total_docs: int = 6,
    extra_source_timeout: float = 3.0,
    max_extra_source_seconds: float = 6.0,
    source_order=None,
    use_mediawiki_rest: bool = False,
    use_perseus_scaife: bool = True,
    use_fordham_sourcebooks: bool = True,
    use_loc: bool = True,
    use_national_archives: bool = False,
    national_archives_api_key: str = "",
    use_europeana: bool = True,
    europeana_api_key: str = "api2demo",
    use_met_collection: bool = True,
    use_reddit: bool = False,
    reddit_client_id: str = "",
    reddit_client_secret: str = "",
    reddit_user_agent: str = "",
    reddit_subreddits=None,
    reddit_top_n_per_subreddit: int = 1,
    reddit_max_docs: int = 4,
    reddit_sort: str = "relevance",
    reddit_time_filter: str = "all",
    reddit_include_comments: bool = False,
    reddit_comment_limit: int = 0,
    perseus_search_kind: str = "form",
    perseus_results_format: str = "instances",
) -> list[dict]:
    """Retrieve Wikipedia docs plus capped documents from history/culture sources."""
    documents = []
    if use_wikipedia and top_n > 0:
        documents.extend(
            get_wikipedia_documents_for_question(
                question,
                top_n=top_n,
                per_query_limit=per_query_limit,
                timeout=timeout,
                max_search_queries=max_search_queries,
            )
        )

    source_order = source_order or ["reddit", "fordham", "loc", "met", "europeana", "perseus", "mediawiki_rest", "national_archives"]
    source_fetchers = {
        "mediawiki_rest": (use_mediawiki_rest, lambda: get_mediawiki_rest_documents_for_question(question, top_n=extra_source_top_n, timeout=extra_source_timeout)),
        "perseus": (use_perseus_scaife, lambda: get_perseus_scaife_documents_for_question(question, top_n=extra_source_top_n, timeout=extra_source_timeout, search_kind=perseus_search_kind, results_format=perseus_results_format)),
        "fordham": (use_fordham_sourcebooks, lambda: get_fordham_sourcebooks_documents_for_question(question, top_n=extra_source_top_n, timeout=extra_source_timeout)),
        "loc": (use_loc, lambda: get_loc_documents_for_question(question, top_n=extra_source_top_n, timeout=extra_source_timeout)),
        "national_archives": (use_national_archives and bool(national_archives_api_key), lambda: get_national_archives_documents_for_question(question, api_key=national_archives_api_key, top_n=extra_source_top_n, timeout=extra_source_timeout)),
        "europeana": (use_europeana and bool(europeana_api_key), lambda: get_europeana_documents_for_question(question, api_key=europeana_api_key, top_n=extra_source_top_n, timeout=extra_source_timeout)),
        "met": (use_met_collection, lambda: get_met_collection_documents_for_question(question, top_n=extra_source_top_n, timeout=extra_source_timeout)),
        "reddit": (use_reddit, lambda: get_reddit_documents_for_question(
            question,
            client_id=reddit_client_id,
            client_secret=reddit_client_secret,
            user_agent=reddit_user_agent,
            subreddits=reddit_subreddits,
            top_n_per_subreddit=reddit_top_n_per_subreddit,
            max_docs=reddit_max_docs,
            sort=reddit_sort,
            time_filter=reddit_time_filter,
            include_comments=reddit_include_comments,
            comment_limit=reddit_comment_limit,
        )),
    }

    extra_start = time.monotonic()
    for source_key in source_order:
        if max_total_docs and len(dedupe_documents(documents)) >= max_total_docs:
            break
        if max_extra_source_seconds is not None and time.monotonic() - extra_start >= max_extra_source_seconds:
            print(f"Extra source budget reached after {time.monotonic() - extra_start:.1f}s; using collected docs.")
            break
        enabled_fetcher = source_fetchers.get(source_key)
        if not enabled_fetcher:
            continue
        enabled, fetcher = enabled_fetcher
        if not enabled:
            continue
        try:
            documents.extend(fetcher())
            documents = dedupe_documents(documents)
        except Exception as exc:
            print(f"Extra source skipped for {source_key!r}: {exc}")

    documents = dedupe_documents(documents)
    if max_total_docs:
        documents = documents[:max_total_docs]
    return documents


# Example after starting a game:
# game = client.game.start(competition_id=comp_id)
# doc = get_wikipedia_document_for_question(game.current_question)
# print(doc["title"])
# print(doc["url"])
# print(doc["text"][:1000])


In [22]:
# Safe retrieval test: by default this does NOT start a timed website game.
# Set USE_LIVE_API_QUESTION_FOR_RETRIEVAL_TEST = True only when you intentionally want a live API question.
USE_LIVE_API_QUESTION_FOR_RETRIEVAL_TEST = False
TOP_N_WIKIPEDIA_DOCS = 3
USE_EXTRA_SOURCES_IN_RETRIEVAL_TEST = True
EXTRA_SOURCE_TOP_N_FOR_TEST = 1
MAX_TOTAL_DOCS_FOR_TEST = 7
EXTRA_SOURCE_TIMEOUT_FOR_TEST = 3.0
MAX_EXTRA_SOURCE_SECONDS_FOR_TEST = 8.0
EUROPEANA_API_KEY_FOR_TEST = "api2demo"  # Replace with your own Europeana key if demo key is rate-limited.
NATIONAL_ARCHIVES_API_KEY_FOR_TEST = ""  # NARA requires an x-api-key, so it stays off unless you add one.
USE_REDDIT_IN_RETRIEVAL_TEST = False
REDDIT_CLIENT_ID_FOR_TEST = ""
REDDIT_CLIENT_SECRET_FOR_TEST = ""
REDDIT_USER_AGENT_FOR_TEST = "python:nlp-entertainment-rag:v1.0 (by /u/YOUR_REDDIT_USERNAME)"
REDDIT_SUBREDDITS_FOR_TEST = ["movies", "television", "gaming", "Music"]
REDDIT_TOP_N_PER_SUBREDDIT_FOR_TEST = 1
REDDIT_MAX_DOCS_FOR_TEST = 4
REDDIT_INCLUDE_COMMENTS_FOR_TEST = False
REDDIT_COMMENT_LIMIT_FOR_TEST = 0

if USE_LIVE_API_QUESTION_FOR_RETRIEVAL_TEST:
    retrieval_test_game = client.game.start(competition_id=comp_id)
    retrieval_test_question = retrieval_test_game.current_question
else:
    retrieval_test_question = "What was considered an important goal of Roman marriage according to the article?"

retrieval_test_docs = get_multi_source_documents_for_question(
    retrieval_test_question,
    top_n=TOP_N_WIKIPEDIA_DOCS,
    per_query_limit=2,
    timeout=4.0,
    max_search_queries=2,
    extra_source_top_n=EXTRA_SOURCE_TOP_N_FOR_TEST,
    max_total_docs=MAX_TOTAL_DOCS_FOR_TEST,
    extra_source_timeout=EXTRA_SOURCE_TIMEOUT_FOR_TEST,
    max_extra_source_seconds=MAX_EXTRA_SOURCE_SECONDS_FOR_TEST,
    use_mediawiki_rest=False,
    use_perseus_scaife=True,
    use_fordham_sourcebooks=True,
    use_loc=True,
    use_national_archives=bool(NATIONAL_ARCHIVES_API_KEY_FOR_TEST),
    national_archives_api_key=NATIONAL_ARCHIVES_API_KEY_FOR_TEST,
    use_europeana=True,
    europeana_api_key=EUROPEANA_API_KEY_FOR_TEST,
    use_met_collection=True,
    use_reddit=USE_REDDIT_IN_RETRIEVAL_TEST,
    reddit_client_id=REDDIT_CLIENT_ID_FOR_TEST,
    reddit_client_secret=REDDIT_CLIENT_SECRET_FOR_TEST,
    reddit_user_agent=REDDIT_USER_AGENT_FOR_TEST,
    reddit_subreddits=REDDIT_SUBREDDITS_FOR_TEST,
    reddit_top_n_per_subreddit=REDDIT_TOP_N_PER_SUBREDDIT_FOR_TEST,
    reddit_max_docs=REDDIT_MAX_DOCS_FOR_TEST,
    reddit_include_comments=REDDIT_INCLUDE_COMMENTS_FOR_TEST,
    reddit_comment_limit=REDDIT_COMMENT_LIMIT_FOR_TEST,
)

docs = retrieval_test_docs

print("=" * 80)
print("RETRIEVAL TEST QUESTION")
print("=" * 80)
print(question_to_text(retrieval_test_question))

print("\nFocused Wikipedia/search queries:")
for query in build_wikipedia_search_queries(retrieval_test_question):
    print("-", query)
print("Extra source query:", build_history_source_query(retrieval_test_question))

print("\n" + "=" * 80)
print(f"TOP {len(retrieval_test_docs)} RELATED DOCUMENTS")
print("=" * 80)

for rank, doc in enumerate(retrieval_test_docs, start=1):
    print(f"\n#{rank} | source={doc.get('source')} | score={doc.get('candidate_score', 0):.3f} | matched query={doc.get('matched_query')!r}")
    print("Title:", doc.get("title"))
    print("URL:", doc.get("url"))
    print("Snippet:", doc.get("snippet", ""))
    print("Preview:", doc.get("text", "")[:500])


RETRIEVAL TEST QUESTION
What was considered an important goal of Roman marriage according to the article?

Focused Wikipedia/search queries:
- Roman marriage
- What was considered an important goal of Roman marriage ?
- What was considered an important goal of Roman marriage according to the article?
Extra source query: Roman marriage

TOP 7 RELATED DOCUMENTS

#1 | source=Wikipedia | score=1.850 | matched query='Roman marriage'
Title: Marriage in ancient Rome
URL: https://en.wikipedia.org/wiki/Marriage_in_ancient_Rome
Snippet: The institution of Roman marriage was a practice of marital monogamy: Roman citizens could have only one spouse at a time in marriage but were allowed to
Preview: Marriage (conubium) was a fundamental institution of society in ancient Rome and was used by Romans primarily as a tool for interfamilial alliances. The institution of Roman marriage was a practice of marital monogamy: Roman citizens could have only one spouse at a time in marriage but were allowed to d

In [23]:
import math


def split_sentences(text: str) -> list[str]:
    """Sentence splitter for clean Wikipedia text."""
    text = normalize_wikipedia_text(text)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [sentence.strip() for sentence in sentences if len(sentence.strip()) >= 40]


def build_rag_chunks(documents: list[dict], sentences_per_chunk: int = 5, overlap: int = 2) -> list[dict]:
    """Split retrieved source documents into overlapping evidence chunks."""
    chunks = []
    step = max(1, sentences_per_chunk - overlap)

    for doc_index, doc in enumerate(documents):
        doc_text = normalize_wikipedia_text(doc.get("text", ""))
        sentences = split_sentences(doc_text)
        if not sentences and len(doc_text) >= 30:
            sentences = [doc_text]
        for start in range(0, len(sentences), step):
            chunk_sentences = sentences[start:start + sentences_per_chunk]
            if not chunk_sentences:
                break
            chunk_text = " ".join(chunk_sentences)
            if len(chunk_text) < 40:
                continue
            chunks.append(
                {
                    "doc_index": doc_index,
                    "chunk_index": len(chunks),
                    "title": doc.get("title", ""),
                    "source": doc.get("source", "Wikipedia"),
                    "url": doc.get("url", ""),
                    "text": chunk_text,
                    "document_score": float(doc.get("candidate_score", 0.0)),
                }
            )
            if start + sentences_per_chunk >= len(sentences):
                break

    return chunks


def lexical_similarity(query: str, text: str) -> float:
    """Fallback score if sklearn is unavailable."""
    query_terms = set(extract_keywords(query, limit=20))
    text_terms = set(tokenize(text))
    if not query_terms or not text_terms:
        return 0.0
    overlap = len(query_terms & text_terms) / len(query_terms)
    return overlap


def retrieve_rag_chunks(question, documents: list[dict], top_k: int = 8) -> list[dict]:
    """Retrieve the strongest chunks from the top-N documents for the question."""
    question_text = question_to_text(question)
    chunks = build_rag_chunks(documents)
    if not chunks:
        return []

    chunk_texts = [f"{chunk.get('source', '')} {chunk['title']} {chunk['text']}" for chunk in chunks]

    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity

        vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        matrix = vectorizer.fit_transform([question_text] + chunk_texts)
        similarities = cosine_similarity(matrix[0:1], matrix[1:]).flatten()
    except Exception:
        similarities = [lexical_similarity(question_text, text) for text in chunk_texts]

    ranked = []
    for chunk, similarity in zip(chunks, similarities):
        score = float(similarity) + 0.08 * chunk.get("document_score", 0.0)
        enriched = dict(chunk)
        enriched["retrieval_score"] = score
        ranked.append(enriched)

    ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return ranked[:top_k]


def build_rag_context(hits: list[dict], max_chars: int = 4200) -> str:
    """Format retrieved chunks as compact evidence for generation."""
    blocks = []
    used = 0
    for index, hit in enumerate(hits, start=1):
        block = f"[Evidence {index} | {hit.get('source', 'Source')}: {hit['title']}] {hit['text']}"
        if used + len(block) > max_chars:
            block = block[:max(0, max_chars - used)]
        if block.strip():
            blocks.append(block)
            used += len(block)
        if used >= max_chars:
            break
    return "\n\n".join(blocks)


def rank_answer_sentences(question, hits: list[dict], max_sentences: int = 5) -> list[str]:
    """Extract the most relevant evidence sentences for a no-LLM answer."""
    question_text = question_to_text(question)
    question_terms = set(tokenize(question_text))
    purpose_terms = {"goal", "purpose", "reason", "important", "considered", "used", "use", "tool", "primarily", "primary", "fundamental", "institution"}
    wants_purpose = bool(question_terms & purpose_terms)
    candidates = []
    seen = set()

    for hit_index, hit in enumerate(hits):
        for sentence_index, sentence in enumerate(split_sentences(hit.get("text", ""))):
            key = sentence.lower()
            if key in seen:
                continue
            seen.add(key)
            sentence_terms = set(tokenize(sentence))
            sentence_for_score = f"{hit.get('source', '')} {hit.get('title', '')} {sentence}"
            score = lexical_similarity(question_text, sentence_for_score) + 0.15 * hit.get("retrieval_score", 0.0)
            if wants_purpose:
                score += 0.25 * len(sentence_terms & purpose_terms)
            if hit.get("title", "").lower() in sentence.lower():
                score += 0.05
            candidates.append((score, hit_index, sentence_index, sentence))

    candidates.sort(key=lambda item: item[0], reverse=True)
    return [sentence for _, _, _, sentence in candidates[:max_sentences]]


def extractive_rag_answer(question, hits: list[dict], max_sentences: int = 5) -> str:
    """Create a concise explanation paragraph from retrieved evidence sentences."""
    sentences = rank_answer_sentences(question, hits, max_sentences=max_sentences)
    if not sentences:
        return "I could not find enough evidence in the retrieved Wikipedia documents to answer confidently."
    return " ".join(sentences)


LLAMA_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
_LLAMA_CACHE = {}


def get_huggingface_token():
    """Read a Hugging Face token from Colab Secrets or environment variables, without hard-coding it."""
    import os

    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        token = os.environ.get(name)
        if token:
            return token

    try:
        from google.colab import userdata
        for name in ("hf_token", "HF_TOKEN", "huggingface"):
            token = userdata.get(name)
            if token:
                return token
    except Exception:
        pass

    return None


def load_llama32_model(model_name: str = LLAMA_MODEL_ID):
    """Load Llama locally. Uses 4-bit quantization on CUDA when available."""
    if model_name in _LLAMA_CACHE:
        return _LLAMA_CACHE[model_name]

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    token = get_huggingface_token()
    tokenizer_kwargs = {"trust_remote_code": True}
    model_kwargs = {"trust_remote_code": True}
    if token:
        tokenizer_kwargs["token"] = token
        model_kwargs["token"] = token

    tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            from transformers import BitsAndBytesConfig
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            model_kwargs.update({"quantization_config": quant_config, "device_map": "auto", "low_cpu_mem_usage": True})
        except Exception:
            model_kwargs.update({"torch_dtype": torch.float16, "device_map": "auto", "low_cpu_mem_usage": True})
    else:
        model_kwargs.update({"torch_dtype": torch.float32, "low_cpu_mem_usage": True})

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if not torch.cuda.is_available():
        model.to("cpu")
    model.eval()

    _LLAMA_CACHE[model_name] = (tokenizer, model)
    return tokenizer, model


def llama32_rag_answer(question, hits: list[dict], model_name: str = LLAMA_MODEL_ID, max_new_tokens: int = 100) -> str:
    """Generate an explanatory RAG answer with the local Llama instruct model."""
    import torch

    tokenizer, model = load_llama32_model(model_name)
    question_text = question_to_text(question)
    context = build_rag_context(hits)

    messages = [
        {
            "role": "system",
            "content": "You answer questions using only the provided evidence. Do not mention multiple-choice options. Write one concise explanatory paragraph. If the evidence is insufficient, say so.",
        },
        {
            "role": "user",
            "content": f"Question: {question_text}\n\nEvidence:\n{context}\n\nAnswer the question in one clear paragraph.",
        },
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer:"

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def answer_question_with_rag(
    question,
    documents: list[dict],
    top_k_chunks: int = 8,
    use_local_generator: bool = True,
    generator_model: str = LLAMA_MODEL_ID,
    generator_max_new_tokens: int = 100,
) -> dict:
    """
    Ask the question from RAG using the top-N documents, without using answer options.

    Returns an explanatory answer plus the retrieved evidence chunks.
    """
    hits = retrieve_rag_chunks(question, documents, top_k=top_k_chunks)
    method = "extractive_rag"

    if use_local_generator:
        try: 
            answer = llama32_rag_answer(question, hits, model_name=generator_model, max_new_tokens=generator_max_new_tokens)
            method = f"llama32_rag:{generator_model}"
        except Exception as exc:
            print(f"Local generator skipped, using extractive RAG instead: {exc}")
            answer = extractive_rag_answer(question, hits)
    else:
        answer = extractive_rag_answer(question, hits)

    return {
        "question": question_to_text(question),
        "answer": answer,
        "method": method,
        "evidence_chunks": hits,
    }


In [24]:
# Run this BEFORE starting a timed game.
# Hugging Face token setup:
# 1. In Colab, open the left sidebar key icon (Secrets).
# 2. Add a secret named HF_TOKEN with your Hugging Face token as the value.
# 3. Enable notebook access for that secret.
# Local fallback: this cell will ask for the token with getpass if no secret/env var is found.

import getpass
import importlib.util
import os
import subprocess
import sys
import time

required_packages = [
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("bitsandbytes", "bitsandbytes"),
    ("praw", "praw"),
]
missing_packages = [package for package, module in required_packages if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

# Optional manual token path. Prefer Colab Secrets named HF_TOKEN.
# If you insist on hardcoding temporarily, put it in MANUAL_HF_TOKEN and run this cell once.
MANUAL_HF_TOKEN = ""

hf_token = get_huggingface_token() or MANUAL_HF_TOKEN.strip()
if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (input hidden): ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token

if not get_huggingface_token():
    raise RuntimeError("No Hugging Face token found. Add HF_TOKEN in Colab Secrets, set MANUAL_HF_TOKEN, or enter it when prompted.")

start_time = time.time()
print(f"Preloading {LLAMA_MODEL_ID} before the timed game...")
llama_tokenizer, llama_model = load_llama32_model(LLAMA_MODEL_ID)

# Small warm-up generation so first real RAG answer does not pay setup cost.
import torch

warmup_messages = [
    {"role": "system", "content": "Answer briefly."},
    {"role": "user", "content": "Say ready."},
]
warmup_prompt = llama_tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
warmup_device = next(llama_model.parameters()).device
warmup_inputs = llama_tokenizer(warmup_prompt, return_tensors="pt").to(warmup_device)
with torch.inference_mode():
    _ = llama_model.generate(
        **warmup_inputs,
        max_new_tokens=2,
        do_sample=False,
        pad_token_id=llama_tokenizer.eos_token_id,
    )

print(f"Llama model is loaded and warmed up in {time.time() - start_time:.1f}s.")
print("Now start the game / run the RAG answer cell.")


Preloading meta-llama/Llama-3.2-3B-Instruct before the timed game...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Llama model is loaded and warmed up in 15.2s.
Now start the game / run the RAG answer cell.


In [25]:
# Test: ask the question from RAG and get an explanatory answer without using answer options.
if "question" not in globals() or question is None:
    game = client.game.start(competition_id=comp_id)
    question = game.current_question

if "docs" not in globals() or not docs:
    docs = get_multi_source_documents_for_question(question, top_n=3, max_total_docs=7)

rag_result = answer_question_with_rag(
    question,
    docs,
    top_k_chunks=8,
    use_local_generator=True,
    generator_model=LLAMA_MODEL_ID,
)

print("=" * 80)
print("QUESTION")
print("=" * 80)
print(rag_result["question"])

print("\n" + "=" * 80)
print("RAG EXPLANATION ANSWER")
print("=" * 80)
print(rag_result["answer"])
print("\nMethod:", rag_result["method"])

print("\n" + "=" * 80)
print("TOP EVIDENCE CHUNKS")
print("=" * 80)
for index, hit in enumerate(rag_result["evidence_chunks"][:3], start=1):
    print(f"\n#{index} | score={hit['retrieval_score']:.3f} | {hit.get('source', 'Source')}: {hit['title']}")
    print(hit["text"][:700])


QUESTION
What was the primary reason the ancient Egyptians developed Egyptian blue?

RAG EXPLANATION ANSWER
There is no clear evidence in the provided sources to suggest that the ancient Egyptians developed Egyptian blue for the purpose of marriage. The provided sources are primarily about the institution of marriage in ancient Rome, its conventions, and its practices, but do not mention the development of Egyptian blue. Egyptian blue is a type of ancient Egyptian pigment, and its development and use are not discussed in the provided sources.

Method: llama32_rag:meta-llama/Llama-3.2-3B-Instruct

TOP EVIDENCE CHUNKS

#1 | score=0.180 | Wikipedia: Marriage in ancient Rome
Conventions of Roman marriage Marriage (conubium) was one of the fundamental institutions of Roman society, as it joined not only two individuals but two families. The Romans considered marriage a partnership, whose primary purpose was to have legitimate descendants to whom property, status, and family qualities could 

In [26]:
import re

LETTERS = "ABCD"


def option_text(option) -> str:
    return option.text if hasattr(option, "text") else option["text"]


def option_id(option) -> int:
    return option.id if hasattr(option, "id") else option["id"]


def parse_option_choice(text: str, option_count: int = 4):
    """Parse A-D or 0-3 from the LLM output."""
    cleaned = str(text).strip().upper()

    letter_match = re.search(r"\b([A-D])\b", cleaned)
    if letter_match:
        index = LETTERS.index(letter_match.group(1))
        return index if index < option_count else None

    digit_match = re.search(r"\b([0-3])\b", cleaned)
    if digit_match:
        index = int(digit_match.group(1))
        return index if index < option_count else None

    return None


def llama_generate_text(messages, model_name: str = LLAMA_MODEL_ID, max_new_tokens: int = 128, max_length: int = 2048) -> str:
    tokenizer, model = load_llama32_model(model_name)
    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer:"

    import torch

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def format_options_for_prompt(options) -> str:
    return "\n".join(
        f"{LETTERS[index]}. {option_text(option)}"
        for index, option in enumerate(options)
    )


def handmade_option_statement(question, option) -> str:
    question_text = question_to_text(question)
    return f'The answer for "{question_text}" is "{option_text(option)}".'


def parse_option_statements(raw_text: str, question, options) -> list[dict]:
    statements_by_letter = {}
    for line in str(raw_text).splitlines():
        match = re.match(r"^\s*(?:[-*]\s*)?([A-D])\s*[\).:\-]\s*(.+?)\s*$", line, flags=re.IGNORECASE)
        if match:
            letter = match.group(1).upper()
            statement = match.group(2).strip()
            statement = re.sub(r"^statement\s*[:\-]\s*", "", statement, flags=re.IGNORECASE).strip()
            if statement:
                statements_by_letter[letter] = statement

    statements = []
    for index, option in enumerate(options):
        letter = LETTERS[index]
        statements.append({
            "letter": letter,
            "option_id": option_id(option),
            "option_text": option_text(option),
            "statement": statements_by_letter.get(letter) or handmade_option_statement(question, option),
        })
    return statements


def build_handmade_option_statements(question, options) -> dict:
    statements = []
    for index, option in enumerate(options):
        statements.append({
            "letter": LETTERS[index],
            "option_id": option_id(option),
            "option_text": option_text(option),
            "statement": handmade_option_statement(question, option),
        })
    return {
        "raw_output": "handmade_template",
        "statements": statements,
    }


def build_realistic_option_statements(question, options, model_name: str = LLAMA_MODEL_ID) -> dict:
    """Use Llama to rewrite each option as a real standalone true/false claim."""
    question_text = question_to_text(question)
    option_lines = format_options_for_prompt(options)
    messages = [
        {
            "role": "system",
            "content": "Rewrite each multiple-choice option as one standalone factual claim that can be judged true or false. Do not say 'the correct answer is'. Do not explain. Keep the option letters A-D.",
        },
        {
            "role": "user",
            "content": f"Question:\n{question_text}\n\nOptions:\n{option_lines}\n\nCreate exactly four standalone true/false claims in this format:\nA. claim\nB. claim\nC. claim\nD. claim",
        },
    ]
    raw_output = llama_generate_text(messages, model_name=model_name, max_new_tokens=180, max_length=2048)
    return {
        "raw_output": raw_output,
        "statements": parse_option_statements(raw_output, question, options),
    }


def build_option_truth_statements(question, options, model_name: str = LLAMA_MODEL_ID, statement_mode: str = "handmade") -> dict:
    mode = str(statement_mode).lower().strip()
    if mode == "handmade":
        return build_handmade_option_statements(question, options)
    if mode == "realistic":
        return build_realistic_option_statements(question, options, model_name=model_name)
    raise ValueError("statement_mode must be 'handmade' or 'realistic'")


def choose_option_from_rag_answer(
    question,
    rag_answer: str,
    options,
    model_name: str = LLAMA_MODEL_ID,
    statement_mode: str = "handmade",
) -> dict:
    """
    Build candidate statements, then ask Llama once to choose the best one.
    statement_mode can be 'handmade' or 'realistic'.
    """
    statement_result = build_option_truth_statements(
        question,
        options,
        model_name=model_name,
        statement_mode=statement_mode,
    )
    statement_lines = "\n".join(
        f"{item['letter']}. {item['statement']}"
        for item in statement_result["statements"]
    )

    messages = [
        {
            "role": "system",
            "content": "Use only the RAG answer and the candidate statements. Pick the statement that is best supported by the RAG answer. Return the option letter first, then a short reason.",
        },
        {
            "role": "user",
            "content": f"RAG answer paragraph:\n{rag_answer}\n\nCandidate true/false statements:\n{statement_lines}\n\nWhich statement is best supported by the RAG answer? Return format: Letter - short reason.",
        },
    ]
    model_output = llama_generate_text(messages, model_name=model_name, max_new_tokens=80, max_length=3072)
    selected_index = parse_option_choice(model_output, option_count=len(options))

    if selected_index is None:
        raise ValueError(f"Could not parse an option choice from Llama output: {model_output!r}")

    selected_option = options[selected_index]
    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_index,
        "letter": LETTERS[selected_index],
        "option_statements": statement_result["statements"],
        "statement_generation_mode": str(statement_mode).lower().strip(),
        "statement_generation_output": statement_result["raw_output"],
        "model_output": model_output,
    }


In [27]:
# Test: convert options into true/false statements, then match them against the RAG paragraph.
if "rag_result" not in globals():
    raise RuntimeError("Run the RAG explanation answer cell first so rag_result exists.")
if "question" not in globals() or question is None:
    raise RuntimeError("No current API question found. Start a game or run the RAG answer cell first.")

option_match = choose_option_from_rag_answer(question, rag_result["answer"], question.options)

print("=" * 80)
print("RAG PARAGRAPH ANSWER")
print("=" * 80)
print(rag_result["answer"])

print("\n" + "=" * 80)
print("OPTIONS")
print("=" * 80)
for index, option in enumerate(question.options):
    print(f"{LETTERS[index]}. [{option_id(option)}] {option_text(option)}")

print("\n" + "=" * 80)
print("GENERATED TRUE/FALSE STATEMENTS")
print("=" * 80)
for item in option_match["option_statements"]:
    print(f"{item['letter']}. {item['statement']}")
print("Statement builder output:", option_match["statement_generation_output"])

print("\n" + "=" * 80)
print("CLOSEST OPTION")
print("=" * 80)
print(f"{option_match['letter']}. [{option_match['answer_id']}] {option_match['answer_text']}")
print("Model output:", option_match["model_output"])


RAG PARAGRAPH ANSWER
There is no clear evidence in the provided sources to suggest that the ancient Egyptians developed Egyptian blue for the purpose of marriage. The provided sources are primarily about the institution of marriage in ancient Rome, its conventions, and its practices, but do not mention the development of Egyptian blue. Egyptian blue is a type of ancient Egyptian pigment, and its development and use are not discussed in the provided sources.

OPTIONS
A. [0] To imitate the precious stones turquoise and lapis lazuli
B. [1] To decorate the exterior of their pyramids
C. [2] To create a new form of glass
D. [3] To produce a pigment for medical purposes

GENERATED TRUE/FALSE STATEMENTS
A. The answer for "What was the primary reason the ancient Egyptians developed Egyptian blue?" is "To imitate the precious stones turquoise and lapis lazuli".
B. The answer for "What was the primary reason the ancient Egyptians developed Egyptian blue?" is "To decorate the exterior of their pyr

In [65]:
# =========================
# ACTUAL GAME: RAG + Llama 3.2 3B + option matching
# =========================
# Run setup/function cells first: imports/login/client, cell 8, cell 10, cell 13.
# This cell preloads Llama BEFORE starting the timed game, then starts the game.

comp_id = 1  # Set your competition ID here.

# ---- Hyperparameters ----
RUN_ACTUAL_GAME = True
COMPETITION_ID = comp_id
MAX_QUESTIONS = None          # Use 1 or 2 for a small test; None = play until game over.
SUBMIT_ANSWERS = True         # True = send answers to API. False = dry run, no submission.

MAX_SEARCH_QUERIES = 1        # Entertainment mode: keep Wikipedia cheap.
PER_QUERY_LIMIT = 2           # Number of Wikipedia search hits per query variant.
TOP_N_DOCS = 1                # Entertainment mode: one Wikipedia doc, then add Reddit docs.

# Extra retrieval sources. Keep these capped; every enabled source spends timed-game seconds.
USE_MEDIAWIKI_REST = False    # Same Wikipedia knowledge via REST; useful as backup, but duplicate-ish.
USE_PERSEUS_SCAIFE = False     # Primary Greek/Latin text snippets from Scaife search.
USE_FORDHAM_SOURCEBOOKS = False
USE_LOC = False             # LOC is available but often slower; turn on if you want it.
USE_NATIONAL_ARCHIVES = False # Requires NATIONAL_ARCHIVES_API_KEY.
USE_EUROPEANA = False          # Uses EUROPEANA_API_KEY; demo key can be rate-limited.
USE_MET_COLLECTION = False
EXTRA_SOURCE_ORDER = ["reddit", "fordham", "met", "europeana", "perseus", "loc", "mediawiki_rest", "national_archives"]
EXTRA_SOURCE_TOP_N = 0        # Docs per extra source.
MAX_TOTAL_DOCS = 5            # 1 Wikipedia doc + up to 4 Reddit docs by default.
EXTRA_SOURCE_TIMEOUT = 2.0    # Seconds per extra-source request.
MAX_EXTRA_SOURCE_SECONDS = 4.0 # Hard budget for non-Wikipedia retrieval.
EUROPEANA_API_KEY = "api2demo" # Replace with your own key if the demo key fails.
NATIONAL_ARCHIVES_API_KEY = "" # NARA requires an x-api-key.
PERSEUS_SEARCH_KIND = "form"
PERSEUS_RESULTS_FORMAT = "instances"

# Reddit source for entertainment questions.
USE_REDDIT = True
REDDIT_CLIENT_ID = ""         # Or set Colab Secret/env var REDDIT_CLIENT_ID.
REDDIT_CLIENT_SECRET = ""     # Or set Colab Secret/env var REDDIT_CLIENT_SECRET.
REDDIT_USER_AGENT = "python:nlp-entertainment-rag:v1.0 (by /u/YOUR_REDDIT_USERNAME)"
REDDIT_SUBREDDITS = ["movies", "television", "gaming", "Music", "popculturechat"]
REDDIT_TOP_N_PER_SUBREDDIT = 1
REDDIT_MAX_DOCS = 4
REDDIT_SEARCH_SORT = "relevance"
REDDIT_TIME_FILTER = "all"
REDDIT_INCLUDE_COMMENTS = False # True is richer but costs extra Reddit API calls/time.
REDDIT_COMMENT_LIMIT = 0

# Rough request budget before early stopping:
# Wikipedia = MAX_SEARCH_QUERIES + min(TOP_N_DOCS, MAX_SEARCH_QUERIES * PER_QUERY_LIMIT)
# Extra sources: fordham<=2, loc<=1, met<=1+EXTRA_SOURCE_TOP_N, europeana<=1, perseus<=1,
# mediawiki_rest<=1+EXTRA_SOURCE_TOP_N, national_archives<=1 when enabled with a key.

WIKIPEDIA_TIMEOUT = 3.0       # Seconds per Wikipedia API request.
WIKIPEDIA_DELAY_SECONDS = 1 # Minimum delay between Wikipedia API requests. Increase if you see 429.
WIKIPEDIA_BACKOFF_SECONDS = 0.5
WIKIPEDIA_RETRIES = 0
TOP_K_CHUNKS = 8              # Number of retrieved evidence chunks passed to Llama.
RAG_MAX_NEW_TOKENS = 300      # Shorter RAG answers are faster and enough for option matching.
STATEMENT_GENERATION_MODE = "handmade"  # "handmade" or "realistic".
QUESTION_TIME_BUFFER = 2.0    # Try to submit before this many seconds remain.
MIN_SECONDS_TO_ATTEMPT = 1.0  # If less time remains, submit fallback option 0.

DELAY_SUBMIT_FOR_WIKI_COOLDOWN = True
TARGET_SUBMIT_ELAPSED_SECONDS = 29  # Leave room for network/server latency.
MIN_SECONDS_LEFT_AT_SUBMIT = 1      # Safety margin for network/server latency.
MAX_SUBMIT_WAIT_SECONDS = 20          # Never wait more than this after answer is ready.

LLAMA_MODEL_FOR_GAME = LLAMA_MODEL_ID
MANUAL_HF_TOKEN = ""          # Prefer Colab Secrets HF_TOKEN. Temporary manual token goes here if needed.
PRELOAD_LLAMA = True
SAVE_RUN_LOG = True
RUN_LOG_DIR = "/content/gdrive/MyDrive/NLP_assignment/test3_rag_game_runs"
VERBOSE = True

# ---- End hyperparameters ----

import getpass
import importlib.util
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def ensure_runtime_packages():
    required_packages = [
        ("transformers", "transformers"),
        ("accelerate", "accelerate"),
        ("bitsandbytes", "bitsandbytes"),
        ("scikit-learn", "sklearn"),
        ("praw", "praw"),
    ]
    missing = [package for package, module in required_packages if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


def ensure_hf_token_for_game():
    token = get_huggingface_token() or MANUAL_HF_TOKEN.strip()
    if not token:
        token = getpass.getpass("Hugging Face token (input hidden): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
    if not get_huggingface_token():
        raise RuntimeError("No Hugging Face token found. Add HF_TOKEN in Colab Secrets, set MANUAL_HF_TOKEN, or enter it when prompted.")


def preload_llama_for_game():
    ensure_runtime_packages()
    ensure_hf_token_for_game()
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
    print(f"Preloading {LLAMA_MODEL_FOR_GAME} before starting timed game...")
    start = time.time()
    tokenizer, model = load_llama32_model(LLAMA_MODEL_FOR_GAME)

    import torch

    warmup_messages = [
        {"role": "system", "content": "Answer briefly."},
        {"role": "user", "content": "Say ready."},
    ]
    warmup_prompt = tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
    device = next(model.parameters()).device
    inputs = tokenizer(warmup_prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        _ = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    print(f"Llama ready in {time.time() - start:.1f}s. Starting game only after this point.")


def seconds_available(game) -> float:
    remaining = game.time_remaining
    if remaining is None:
        return 30.0
    return max(0.0, float(remaining))


def fallback_option(question):
    return question.options[0]


def wait_before_submit_for_cooldown(game) -> float:
    """Wait after computing the answer so Wikipedia gets cooldown time before next question."""
    if not DELAY_SUBMIT_FOR_WIKI_COOLDOWN:
        return 0.0

    current_remaining = seconds_available(game)
    target_remaining = max(MIN_SECONDS_LEFT_AT_SUBMIT, 30.0 - TARGET_SUBMIT_ELAPSED_SECONDS)
    wait_seconds = current_remaining - target_remaining
    wait_seconds = min(MAX_SUBMIT_WAIT_SECONDS, max(0.0, wait_seconds))

    if wait_seconds > 0:
        print(f"Answer ready. Waiting {wait_seconds:.1f}s before submit to give Wikipedia API cooldown time.")
        time.sleep(wait_seconds)

    return wait_seconds


def answer_one_question_with_pipeline(question, game=None) -> dict:
    start = time.monotonic()
    seconds_left_start = seconds_available(game) if game is not None else None

    docs = get_multi_source_documents_for_question(
        question,
        top_n=TOP_N_DOCS,
        per_query_limit=PER_QUERY_LIMIT,
        timeout=WIKIPEDIA_TIMEOUT,
        max_search_queries=MAX_SEARCH_QUERIES,
        extra_source_top_n=EXTRA_SOURCE_TOP_N,
        max_total_docs=MAX_TOTAL_DOCS,
        extra_source_timeout=EXTRA_SOURCE_TIMEOUT,
        max_extra_source_seconds=MAX_EXTRA_SOURCE_SECONDS,
        source_order=EXTRA_SOURCE_ORDER,
        use_mediawiki_rest=USE_MEDIAWIKI_REST,
        use_perseus_scaife=USE_PERSEUS_SCAIFE,
        use_fordham_sourcebooks=USE_FORDHAM_SOURCEBOOKS,
        use_loc=USE_LOC,
        use_national_archives=USE_NATIONAL_ARCHIVES,
        national_archives_api_key=NATIONAL_ARCHIVES_API_KEY,
        use_europeana=USE_EUROPEANA,
        europeana_api_key=EUROPEANA_API_KEY,
        use_met_collection=USE_MET_COLLECTION,
        use_reddit=USE_REDDIT,
        reddit_client_id=REDDIT_CLIENT_ID,
        reddit_client_secret=REDDIT_CLIENT_SECRET,
        reddit_user_agent=REDDIT_USER_AGENT,
        reddit_subreddits=REDDIT_SUBREDDITS,
        reddit_top_n_per_subreddit=REDDIT_TOP_N_PER_SUBREDDIT,
        reddit_max_docs=REDDIT_MAX_DOCS,
        reddit_sort=REDDIT_SEARCH_SORT,
        reddit_time_filter=REDDIT_TIME_FILTER,
        reddit_include_comments=REDDIT_INCLUDE_COMMENTS,
        reddit_comment_limit=REDDIT_COMMENT_LIMIT,
        perseus_search_kind=PERSEUS_SEARCH_KIND,
        perseus_results_format=PERSEUS_RESULTS_FORMAT,
    )
    after_retrieval = time.monotonic()

    rag_result = answer_question_with_rag(
        question,
        docs,
        top_k_chunks=TOP_K_CHUNKS,
        use_local_generator=True,
        generator_model=LLAMA_MODEL_FOR_GAME,
        generator_max_new_tokens=RAG_MAX_NEW_TOKENS,
    )
    after_rag = time.monotonic()

    option_match = choose_option_from_rag_answer(
        question,
        rag_result["answer"],
        question.options,
        model_name=LLAMA_MODEL_FOR_GAME,
        statement_mode=STATEMENT_GENERATION_MODE,
    )
    after_match = time.monotonic()

    elapsed = after_match - start
    seconds_left_end = seconds_available(game) if game is not None else None

    return {
        "question": question.text,
        "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
        "documents": [
            {
                "source": doc.get("source"),
                "title": doc.get("title"),
                "url": doc.get("url"),
                "candidate_score": doc.get("candidate_score"),
                "matched_query": doc.get("matched_query"),
                "preview": doc.get("text", "")[:500],
            }
            for doc in docs
        ],
        "rag_answer": rag_result["answer"],
        "rag_method": rag_result["method"],
        "evidence_chunks": [
            {
                "source": hit.get("source"),
                "title": hit.get("title"),
                "retrieval_score": hit.get("retrieval_score"),
                "text": hit.get("text", "")[:900],
            }
            for hit in rag_result["evidence_chunks"][:5]
        ],
        "option_match": option_match,
        "elapsed_seconds": elapsed,
        "timings": {
            "retrieval_seconds": after_retrieval - start,
            "wikipedia_seconds": after_retrieval - start,
            "rag_generation_seconds": after_rag - after_retrieval,
            "option_matching_seconds": after_match - after_rag,
            "pre_submit_pipeline_seconds": elapsed,
        },
        "seconds_left_start": seconds_left_start,
        "seconds_left_end": seconds_left_end,
    }


def play_actual_rag_game():
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES

    if PRELOAD_LLAMA:
        preload_llama_for_game()

    if not RUN_ACTUAL_GAME:
        print("RUN_ACTUAL_GAME is False. Set it to True to start a real timed game.")
        return None, None

    game = client.game.start(competition_id=COMPETITION_ID)
    run_log = {
        "session_id": game.session_id,
        "competition_id": COMPETITION_ID,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "top_n_docs": TOP_N_DOCS,
            "per_query_limit": PER_QUERY_LIMIT,
            "max_search_queries": MAX_SEARCH_QUERIES,
            "wikipedia_timeout": WIKIPEDIA_TIMEOUT,
            "wikipedia_delay_seconds": WIKIPEDIA_DELAY_SECONDS,
            "wikipedia_backoff_seconds": WIKIPEDIA_BACKOFF_SECONDS,
            "wikipedia_retries": WIKIPEDIA_RETRIES,
            "extra_sources": {
                "use_mediawiki_rest": USE_MEDIAWIKI_REST,
                "use_perseus_scaife": USE_PERSEUS_SCAIFE,
                "use_fordham_sourcebooks": USE_FORDHAM_SOURCEBOOKS,
                "use_loc": USE_LOC,
                "use_national_archives": USE_NATIONAL_ARCHIVES,
                "use_europeana": USE_EUROPEANA,
                "use_met_collection": USE_MET_COLLECTION,
                "use_reddit": USE_REDDIT,
                "reddit_subreddits": REDDIT_SUBREDDITS,
                "reddit_top_n_per_subreddit": REDDIT_TOP_N_PER_SUBREDDIT,
                "reddit_max_docs": REDDIT_MAX_DOCS,
                "reddit_search_sort": REDDIT_SEARCH_SORT,
                "reddit_time_filter": REDDIT_TIME_FILTER,
                "reddit_include_comments": REDDIT_INCLUDE_COMMENTS,
                "reddit_comment_limit": REDDIT_COMMENT_LIMIT,
                "reddit_client_id_set": bool(REDDIT_CLIENT_ID) or bool(get_secret_value("REDDIT_CLIENT_ID")),
                "extra_source_order": EXTRA_SOURCE_ORDER,
                "extra_source_top_n": EXTRA_SOURCE_TOP_N,
                "max_total_docs": MAX_TOTAL_DOCS,
                "extra_source_timeout": EXTRA_SOURCE_TIMEOUT,
                "max_extra_source_seconds": MAX_EXTRA_SOURCE_SECONDS,
                "europeana_api_key_set": bool(EUROPEANA_API_KEY),
                "national_archives_api_key_set": bool(NATIONAL_ARCHIVES_API_KEY),
            },
            "top_k_chunks": TOP_K_CHUNKS,
            "rag_max_new_tokens": RAG_MAX_NEW_TOKENS,
            "statement_generation_mode": STATEMENT_GENERATION_MODE,
            "question_time_buffer": QUESTION_TIME_BUFFER,
            "min_seconds_to_attempt": MIN_SECONDS_TO_ATTEMPT,
            "delay_submit_for_wiki_cooldown": DELAY_SUBMIT_FOR_WIKI_COOLDOWN,
            "target_submit_elapsed_seconds": TARGET_SUBMIT_ELAPSED_SECONDS,
            "min_seconds_left_at_submit": MIN_SECONDS_LEFT_AT_SUBMIT,
            "max_submit_wait_seconds": MAX_SUBMIT_WAIT_SECONDS,
            "llama_model": LLAMA_MODEL_FOR_GAME,
            "submit_answers": SUBMIT_ANSWERS,
        },
        "questions": [],
    }

    print(f"Started game session {game.session_id}. Competition {COMPETITION_ID}.")
    question_count = 0
    correct_count = 0

    while game.in_progress:
        question = game.current_question
        if question is None:
            print("No active question returned by server.")
            break

        question_count += 1
        current_level = game.current_level
        time_left = seconds_available(game)

        print("\n" + "=" * 80)
        print(f"Question {question_count} | Level {current_level} | {time_left:.1f}s left")
        print(question.text)
        for index, opt in enumerate(question.options):
            print(f"  {LETTERS[index]}. [{option_id(opt)}] {option_text(opt)}")
        print("=" * 80)

        if time_left < MIN_SECONDS_TO_ATTEMPT:
            selected = fallback_option(question)
            prediction = {
                "question": question.text,
                "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
                "documents": [],
                "rag_answer": "Skipped RAG because not enough time remained.",
                "rag_method": "fallback_time_guard",
                "evidence_chunks": [],
                "option_match": {
                    "answer_id": option_id(selected),
                    "answer_text": option_text(selected),
                    "answer_index": 0,
                    "letter": "A",
                    "model_output": "fallback_time_guard",
                },
                "elapsed_seconds": 0.0,
                "seconds_left_start": time_left,
                "seconds_left_end": time_left,
            }
        else:
            prediction = answer_one_question_with_pipeline(question, game=game)

        selected_id = prediction["option_match"]["answer_id"]
        selected_text = prediction["option_match"]["answer_text"]
        selected_letter = prediction["option_match"]["letter"]

        if VERBOSE:
            print("\nRAG answer:")
            print(prediction["rag_answer"])
            print("\nGenerated true/false statements:")
            print("Statement mode:", prediction["option_match"].get("statement_generation_mode"))
            for item in prediction["option_match"].get("option_statements", []):
                print(f"  {item['letter']}. {item['statement']}")
            print("\nClosest option:", f"{selected_letter}. [{selected_id}] {selected_text}")
            print("Matcher output:", prediction["option_match"].get("model_output"))
            timings = prediction.get("timings", {})
            if timings:
                print(
                    "Timing:",
                    f"retrieval={timings.get('retrieval_seconds', timings.get('wikipedia_seconds', 0)):.2f}s",
                    f"rag={timings.get('rag_generation_seconds', 0):.2f}s",
                    f"match={timings.get('option_matching_seconds', 0):.2f}s",
                    f"total={timings.get('pre_submit_pipeline_seconds', prediction['elapsed_seconds']):.2f}s",
                )
            print(f"Elapsed: {prediction['elapsed_seconds']:.2f}s | Time left now: {seconds_available(game):.1f}s")

        result_payload = None
        if SUBMIT_ANSWERS:
            submission_wait_seconds = wait_before_submit_for_cooldown(game)
            prediction["submission_wait_seconds"] = submission_wait_seconds
            if submission_wait_seconds:
                print(f"Time left after cooldown wait: {seconds_available(game):.1f}s")

            if seconds_available(game) <= QUESTION_TIME_BUFFER:
                print("Warning: low time before submit; submitting selected option immediately.")
            result = game.answer(selected_id)
            result_payload = {
                "correct": result.correct,
                "timed_out": result.timed_out,
                "game_over": result.game_over,
                "earned_amount": result.earned_amount,
            }
            correct_count += int(bool(result.correct))

            if result.correct:
                print(f"Correct. Earned: {result.earned_amount}")
            elif result.timed_out:
                print(f"Timed out. Earned: {result.earned_amount}")
            else:
                print(f"Wrong. Earned: {result.earned_amount}")
        else:
            print("Dry run: answer not submitted.")

        run_log["questions"].append(
            {
                "number": question_count,
                "level": current_level,
                "prediction": prediction,
                "result": result_payload,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )

        if result_payload and result_payload.get("game_over"):
            break
        if MAX_QUESTIONS is not None and question_count >= MAX_QUESTIONS:
            print("MAX_QUESTIONS reached; stopping.")
            break
        if not SUBMIT_ANSWERS:
            break

    run_log["finished_at"] = datetime.now(timezone.utc).isoformat()
    run_log["questions_answered"] = question_count
    run_log["correct_count"] = correct_count
    run_log["final_earned_amount"] = game.earned_amount

    if SAVE_RUN_LOG:
        log_dir = Path(RUN_LOG_DIR)
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f"test3_rag_game_{game.session_id}.json"
        with open(log_path, "w", encoding="utf-8") as handle:
            json.dump(run_log, handle, indent=2, ensure_ascii=False)
        print("Run log saved to:", log_path)

    print("\nGame summary")
    print("Questions answered:", question_count)
    print("Correct answers:", correct_count)
    print("Final earnings:", game.earned_amount)
    return game, run_log


final_game, final_run_log = play_actual_rag_game()


Preloading meta-llama/Llama-3.2-3B-Instruct before starting timed game...
Llama ready in 0.3s. Starting game only after this point.
Started game session 78794. Competition 1.

Question 1 | Level 1 | 29.9s left
How did the Neo-Assyrian Empire ensure the loyalty of its conquered territories?
  A. [0] By imposing a common language and religion
  B. [1] By relocating entire populations to new regions
  C. [2] By destroying local cultures and identities
  D. [3] By allowing local rulers to maintain power as vassals

RAG answer:
The Neo-Assyrian Empire ensured the loyalty of its conquered territories through a combination of military campaigns, administrative structures, and cultural assimilation. The empire's extensive military campaigns, such as those led by Tiglath-Pileser III and Sennacherib, helped to establish a network of vassal states and provinces, with the empire's kings often incorporating the local populations into their administrative structures. The empire's administrative stru